> **仓库整理提示｜2026-11-23**：本篇来自已保存的中文学习版，不保证包含个人电脑上的后续修改。本次未重新执行真实数据；保留输出仅作示例，不是本次复现记录。
>
> 先阅读 [运行指南](../../guides/setup.md) 和 [版本来源](SOURCES.md)。从前往后执行，辅助函数必须先定义再调用。下面开始原中文学习版正文。


> **中文版说明**：本 Notebook 译自 [xdas-dev/tutorials 的 01_open_and_consolidate.ipynb](https://github.com/xdas-dev/tutorials/blob/main/01_open_and_consolidate.ipynb)，原项目采用 [GPL-3.0 许可证](https://github.com/xdas-dev/tutorials/blob/main/LICENSE.md)。保留原代码并翻译说明，另外添加两处明确标注的辅助单元格，以适配本次安装在台站下载处中断的情况。请在包含 `data/` 和 `outputs/` 的教程运行根目录启动，路径不依赖特定盘符。

# 打开并整合 DAS 数据

## 先认识这份数据

DAS 解调仪沿光纤采集信号：可以把一根光纤看作许多按距离排列的**通道**，每个通道都随时间产生测量值。于是，一段记录可表示为一个 `time × distance` 的二维数据数组。真实采集往往切成许多短文件，并按解调仪和光缆分别存放。

本例使用智利中部 **ABYSS 实验**提供的约十分钟样本。两台解调仪涉及三条光缆记录：`CCN/N`、`SER/N`、`SER/S`（`N` / `S` 表示向北 / 向南）；每条记录约每十秒生成一个 HDF5 文件。文件中的 `distance` 是沿光纤的位置，真实经纬度需要从另外的光缆路线 CSV 补充。下面用到的文件已在 `data/gps_multicable/`，路线文件位于 `data/geometry/`。

## 本教程的目的

把分散的 HDF5 时间片整理成便于分析的结构：先把同一根光纤的文件沿时间轴**虚拟连接**，再把多根光纤放进一棵树，补上通道的地理坐标，最后保存供后续教程使用的入口文件。虚拟连接保留对原始 HDF5 的引用，读取某段数据时才访问所需样本。完成后你应能解释 `DataArray`、`DataCollection`、`VirtualSource` 和 `VirtualStack` 各表示什么。

**本教程的路线**

| 步骤 | 要解决的问题 |
| --- | --- |
| 1. 单个文件 | 看懂一段约十秒记录的时间、通道与虚拟引用。 |
| 2–3. 多个文件 | 把数个文件乃至一次完整采集连接为一个按时间索引的数组。 |
| 4. 多层结构 | 按解调仪、光缆和采集段组织不同形状的数组。 |
| 5. 地理位置 | 将光缆测绘的经纬度对应到通道。 |
| 6. 保存成果 | 生成 `outputs/multicable.nc` 和 `outputs/singlecable.nc`，供后续 Notebook 打开。 |
| 7. 参数变化 | 观察形状不一致的文件如何分组成多个采集段。 |

> 运行方式：在 教程运行根目录（包含 `data/` 和 `outputs/`） 下打开本 Notebook，确认 Kernel 是本项目的 Python 环境。按 **Shift + Enter** 逐格运行；后面的单元格会用到前面生成的 `da` 和 `dc`。

In [ ]:
import xdas as xd

先看看这份数据在磁盘上的目录，以及 `SER/N` 中前三个时间片的文件名。
> **Windows 兼容说明：** 官方 `%ls` 在某些 Windows/IPython 环境会把 `/` 当作命令选项；下一格已改为标准 Python 的 `pathlib` 列目录，结果用途相同。

In [ ]:
from pathlib import Path

root = Path("data/gps_multicable")
print("数据目录：")
for item in sorted(root.iterdir()):
    print(item.name)

print("\nSER/N 中前 3 个 HDF5 文件：")
for item in sorted((root / "SER" / "N").glob("*.hdf5"))[:3]:
    print(item.name)


## 1. 打开单个文件

把普通文件路径交给 `xd.open`，会得到一个 `DataArray`。XDAS 会根据文件本身识别格式，通常不必手动指定读取方式；只有无法自动识别时才需要传入 `engine`。

In [ ]:
xd.open("data/gps_multicable/SER/N/122001.hdf5")

### 如何读懂上一个单元格的输出

`xd.open("data/gps_multicable/SER/N/122001.hdf5")` 打开的是 **一个文件**。你截图中看到的内容是 XDAS 对这个文件的概览：

| 输出 | 含义 |
| --- | --- |
| `xdas.DataArray (time: 625, distance: 10000)` | 这是一个二维数组：时间轴有 625 个采样点，距离轴有 10,000 个通道。每个时间点都对应这 10,000 个通道的一组测量值。 |
| `VirtualSource: 11.9MB (int16)` | 原始数值采用 16 位整数；约 11.9 MB 是这块数据的大小。`VirtualSource` 表示目前保留的是指向 HDF5 原文件的引用，尚未把全部波形读进内存。 |
| `time: 2023-11-03T12:20:01.000 to 2023-11-03T12:20:10.984` | 该文件覆盖约 10 秒。首尾时间戳之差为 9.984 秒，因为 625 个采样点之间只有 624 个采样间隔。 |
| `distance: 0.000 to 153179.709` | 通道对应的沿光纤距离从 0 到约 153.18 km；这里的距离坐标不是经纬度，也不等于两点之间的直线距离。 |

上方仅显示两个坐标轴的**范围**，不是把 625 × 10,000 个测量值全部打印出来。需要实际数值时，XDAS 才会从源文件读取相应样本。

**与下一节的关系：** 这里展示的是“打开单个文件”。下一节“打开少量文件”会同时打开 `122001.hdf5`、`122011.hdf5` 和 `122021.hdf5`，把三个约 10 秒的文件沿时间轴连接。

## 2. 打开少量文件

传入路径列表后，文件会沿第一个维度连接；这里的第一个维度是时间，因此三个十秒文件合成一段连续记录：

In [ ]:
xd.open(
    [
        "data/gps_multicable/SER/N/122001.hdf5",
        "data/gps_multicable/SER/N/122011.hdf5",
        "data/gps_multicable/SER/N/122021.hdf5",
    ]
)

现在得到约三十秒的数据。表示中的 `VirtualStack` 表明各文件的指针被堆叠起来；与单文件的 `VirtualSource` 对照看。若连接轴不是第一个维度，可用 `dim` 指定。

XDAS 在连接时会检查坐标。本例使用 GPS 同步，时间恰好按采样点衔接；时间戳较不可靠时，`tolerance` 用于设定允许多大误差仍视为连续。第 04 份教程会详细讨论。

## 3. 打开一次完整采集

文件一多，手写列表就不方便了。通配符 `*`、`?` 或 `[…]` 可以匹配并打开一批文件：

In [ ]:
da = xd.open("data/gps_multicable/SER/N/*.hdf5")
da

这里的六十个文件合成一个约十分钟的数组。打开时主要是逐个读取元数据，因此即使有上千文件也可以处理；规模更大时可用 `parallel` 并行扫描，用 `verbose=True` 显示进度。例如，把上一个单元格改成：

```python
da = xd.open(
    "data/gps_multicable/SER/N/*.hdf5",
    parallel=4,       # 用 4 个进程并行读取各文件的元数据
    verbose=True,     # 显示扫描进度条
)
da
```

`parallel=4` 中的数字是进程数，可以按电脑负载调整；`parallel=True` 使用可用的处理器核心数，`parallel=False` 则顺序处理。这一节只有 60 个文件，无需为了完成教程而开启并行；这里主要展示文件数量更大时的写法。

接下来你可以把它当作一个数组，按物理坐标选择时间段与距离段。和 xarray 一样，范围选择要用 `slice`。

**观察点：** `da.sel(...)` 返回的是整个采集中指定的时空窗口，并没有要求你先判断它跨越了哪些原始文件。

In [ ]:
da.sel(
    time=slice("2023-11-03T12:26:40", "2023-11-03T12:27:50"),
    distance=slice(20_000, 100_000),
)

## 4. 打开完整实验的多层结构

一个 `DataArray` 对应一个数组。但当多个数组形状不同时，例如三根光纤同时采集，或采样参数中途改变，就需要 `DataCollection` 容纳它们。它是树形结构：节点可以是类似 `dict` 的 `DataMapping`，也可以是类似 `list` 的 `DataSequence`。

本例目录依次表示解调仪所在节点、光纤方向（`N` 为北、`S` 为南），再往下是记录文件。官方示例使用 `xd.open("data/gps_multicable/{node}/{cable}/[record].hdf5")`：花括号标记字典层级，方括号标记列表层级。

> **Windows 兼容说明：** 在当前环境中，该路径模板可能在匹配 Windows 文件路径时触发 `AttributeError: 'NoneType' object has no attribute 'group'`。下面用公开的 `xd.DataCollection` 接口组装同一棵树；沿用上一节已打开的 `da`，后续的 `dc["SER"]["N"][0]`、`query`、`sel` 和保存代码可以继续运行。

In [ ]:
dc = xd.DataCollection(
    {
        "CCN": ("cable", {
            "N": ("record", [xd.open("data/gps_multicable/CCN/N/*.hdf5")]),
        }),
        "SER": ("cable", {
            "N": ("record", [da]),  # 沿用上一节已打开的完整采集
            "S": ("record", [xd.open("data/gps_multicable/SER/S/*.hdf5")]),
        }),
    },
    name="node",
)
dc

`CCN` 节点有一根向北的光纤；`SER` 节点有向北、向南两根。每根光纤在这十分钟内的采集参数未变，因此这里只包含一次采集；若参数中途改变，`record` 列表就会有更多项。

用 `plot_availability` 可以快速检查：各条光纤在哪些时段有数据，是否和预期一致。

In [ ]:
xd.plot_availability(dc, dim="time", height=300)

### 逐层访问树

和嵌套的字典、列表一样，可以逐层使用索引访问：

In [ ]:
dc["SER"]["N"][0]

树较深时，`query` 更方便：它按**层级名称**而非位置访问，还能使用通配符。

In [ ]:
dc.query(node="S*")

指定多个层级，就能一路筛选到某个叶节点：

In [ ]:
dc.query(node="SER", cable="N")

### 对树中的每个数组执行操作

`query` 根据层级位置决定**保留哪些叶节点**；`sel` 则根据坐标值裁剪**每个叶节点内部的数据**。像其他适合逐元素应用的 `DataArray` 方法一样，`sel` 会应用到整棵树的叶节点。

In [ ]:
dc.sel(time=slice("2023-11-03T12:26:40", "2023-11-03T12:27:50"))

## 5. 给通道补上光缆地理位置

解调仪文件里通常没有每个通道的真实经纬度，需要利用敷设时测量的光缆路线。本例的路线已经整理为 `data/geometry/{node}_{cable}.csv`，其中包含沿缆距离、纬度和经度。

如果在保存汇总文件前补上这些坐标，后续 Notebook 就能直接使用，不必各自重新插值。

> **你的安装状态：** `install.py` 已经解压 DAS 数据，但在获取 EarthScope 台站资料时中断，因此后面的 `data/geometry/` 可能尚未生成。先运行下一格“补齐几何文件”；成功后再继续本节。它只补几何 CSV，不重新下载 3.6 GB DAS 压缩包。

In [ ]:
from pathlib import Path

# 中文版补充：仅在几何 CSV 缺失时下载它们，不重新下载 DAS 波形。
required = [Path(f"data/geometry/{name}.csv") for name in ("CCN_N", "SER_N", "SER_S")]
if not all(path.exists() for path in required):
    import install
    install.fetch_geometry(False)
print("几何文件已就绪：", all(path.exists() for path in required))


In [ ]:
import numpy as np
import pandas as pd


def get_cable_coords(da, node, cable):
    """Interpolate a cable's surveyed route onto the channels of `da`."""
    geometry = pd.read_csv(f"data/geometry/{node}_{cable}.csv")
    distance = geometry["distance"].to_numpy() * 1000  # km -> m, like `da`
    return tuple(
        np.interp(
            da["distance"].values,
            distance,
            geometry[column].to_numpy(),
            left=np.nan,
            right=np.nan,
        )
        for column in ("latitude", "longitude")
    )

每个叶节点对应各自的光缆路线。对每根光纤插值一次，把 `latitude` 和 `longitude` 作为沿 `distance` 维度变化的两个新坐标附上：

In [ ]:
for node in dc:
    for cable in dc[node]:
        da = dc[node][cable][0]
        latitude, longitude = get_cable_coords(da, node, cable)
        dc[node][cable][0] = da.assign_coords(
            latitude=("distance", latitude),
            longitude=("distance", longitude),
        )

dc["SER"]["N"][0]

这里有两点值得留意。`SER/N` 的实测路线只到 102 km，而解调仪覆盖到 153 km；超出实测范围的最后约三分之一通道被赋为 `NaN`，没有编造位置。另外，前 4.6 km 的通道对应同一个坐标，因为这段光纤盘绕在岸站，通道序号连续而空间位置几乎不变。

**物理提醒：** 沿光纤距离不等于两点之间的直线距离。

In [ ]:
lat = dc["SER"]["N"][0]["latitude"].values
print(f"{np.isnan(lat).sum()} of {len(lat)} SER/N channels have no mapped position")

## 6. 保存“虚拟连接”

汇总视图写入磁盘时不会复制全部 DAS 样本；生成的文件主要保存指向原始数据的引用。这些汇总文件可供后续 Notebook 读取，也方便团队共享。

**注意：** 引用仍依赖原始 HDF5 文件的位置。以后若移动或删除 `data/`，保存的虚拟汇总文件可能无法找到波形。

> **中文版补充：** 下一个单元格会确认 `outputs/` 目录存在，再保存虚拟汇总文件。

In [ ]:
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)


In [ ]:
dc.to_netcdf("outputs/multicable.nc")  # virtual=True by default
dc["SER"]["N"][0].to_netcdf("outputs/singlecable.nc")

再次打开汇总文件，仍然使用 `xd.open`。XDAS 会识别文件里是单个数组还是集合，无需换另一种读取命令。

In [ ]:
xd.open("outputs/multicable.nc")

In [ ]:
xd.open("outputs/singlecable.nc")

## 7. 文件形状不一致时

若文件块的形状不同，它们不能全部并成一个数组。XDAS 会把兼容的部分分组，返回一个序列，而不是直接报错。

In [ ]:
xd.open("data/gps_multiacq/*.hdf5")

这里有三次采集，即采样参数发生了两次变化。读取各段的采样间隔，看看**时间采样间隔 `dt`** 和 **通道间距 `dx`** 分别如何变化：

In [ ]:
print("#   dt      dx")
print("-----------------")
for acquisition, da in enumerate(xd.open("data/gps_multiacq/*.hdf5")):
    dt = xd.get_sampling_interval(da, "time")
    dx = xd.get_sampling_interval(da, "distance")
    print(f"{acquisition}   {dt * 1000:>2.0f} ms   {dx:>5.2f} m")

> **总结**
>
> 通配符打开多个文件时，结果会随数据结构而变：全部兼容就得到一个 `DataArray`；不兼容则得到 `DataCollection`。如果希望始终得到集合、减少后续判断对象类型的分支，可传入 `squeeze=False`。
>
> **完成检查：** 你能否解释 `VirtualSource` 与 `VirtualStack` 的差别？能否从 `DataCollection` 中选出 `SER/N`，并说明 `sel` 与 `query` 各筛选了什么？